# 1. Install & Import Dependencies
This section ensures required packages are installed and brings in all main Python libraries.

In [1]:
# Install specific numpy version for reproducibility
%pip install numpy==2.2

In [2]:
# Standard data science imports
import pandas as pd  # Tabular data handling
import numpy as np   # Numerical computations

# 2. Download & Load Dataset from OpenML
Fetches the Counter-Strike dataset from OpenML using HTTP request.

In [3]:
import requests  # For downloading external data

# OpenML public dataset URL (.arff file containing match rounds)
url = "https://www.openml.org/data/download/22102255/dataset"

# Make GET request to download the data
r = requests.get(url, allow_redirects=True)

In [4]:
# Save ARFF dataset as local text file for parsing
with open("dataset.txt", "wb") as f:
    f.write(r.content)

# 3. Parse ARFF Format & Save As CSV
Extracts tabular rows and column names from ARFF, skips metadata/comments.

In [5]:
# ARFF parser: skip metadata lines, keep only data rows
data = []
with open("dataset.txt", "r") as f:
    for line in f.read().split("\n"):
        if line.startswith("@") or line.startswith("%") or line == "":
            continue  # Ignore metadata and empty lines
        data.append(line)
# 'data' now holds only CSV-style round rows

In [6]:
# Extract column names from ARFF header (@ATTRIBUTE lines)
columns = []
with open("dataset.txt", "r") as f:
    for line in f.read().split("\n"):
        if line.startswith("@ATTRIBUTE"):
            columns.append(line.split()[1])  # 2nd token is usually column name

In [7]:
# Combine columns header and all data rows into a fresh CSV file
with open("df.csv", "w") as f:
    f.write(",".join(columns))  # Header
    f.write("\n")
    f.write("\n".join(data))    # All rounds

# 4. Load & Explore Data (EDA)
Loading, inspecting column types, class balance, sample rounds etc.

In [8]:
# Load cleaned CSV into Pandas DataFrame
df = pd.read_csv("df.csv")
df.columns = columns  # Ensures proper dtype and order

In [9]:
# Optional: Pandas option to show all columns when printing DataFrames
pd.set_option('display.max_columns', None)
# Uncomment the next line if you also want to see all rows (may be slow for large datasets)
# pd.set_option('display.max_rows', None)

# Preview the first few rows of the dataset
df.head()

# Distribution of rounds by map — check balance between maps
df['map'].value_counts()

# Distribution of target labels — check class balance between CT and T round wins
df['round_winner'].value_counts()

# 5. Preprocessing – One‑Hot Encode the 'map' column
Converts the text/categorical 'map' column into binary indicator columns for ML models.

In [10]:
df = pd.get_dummies(df, columns=['map'])  # Creates columns like 'map_de_dust2', 'map_de_inferno', etc.

# 6. Feature / Target Split
Separates inputs (X) from outputs/labels (y).

In [11]:
X = df.drop(columns=['round_winner'])  # Features
y = df['round_winner']                 # Target label

# 7. Train–Test Split
Creates separate datasets for training and final evaluation.

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y  # maintain class ratio
)

# 8. Model Training – Logistic Regression
A linear classifier baseline for performance comparison.

In [13]:
from sklearn.linear_model import LogisticRegression
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

# Evaluate model performance on test set
logreg_acc = log_reg.score(X_test, y_test)
print("Logistic Regression Accuracy:", logreg_acc)

# 9. Model Training – K‑Nearest Neighbors (with RandomizedSearchCV)
Non‑parametric model whose performance depends on distance metric and number of neighbors.

In [14]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RandomizedSearchCV

knn = KNeighborsClassifier(n_jobs=-1)

# Hyperparameter grid
param_dist = {
    "n_neighbors": [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23],
    "weights": ["uniform", "distance"]
}

# Randomized search for efficiency with limited trials
knn_search = RandomizedSearchCV(
    knn,
    param_distributions=param_dist,
    n_iter=3,
    cv=3,
    n_jobs=-1,
    verbose=2
)

knn_search.fit(X_train, y_train)
best_knn = knn_search.best_estimator_
knn_acc = best_knn.score(X_test, y_test)
print("Best KNN Accuracy:", knn_acc)

# 10. Model Training – Random Forest Classifier
Ensemble method using many decision trees; robust to irrelevant features.

In [15]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)

rf_acc = rf.score(X_test, y_test)
print("Random Forest Accuracy:", rf_acc)

# 11. Save Best Model
Stores the trained RandomForest model to reuse without retraining.

In [16]:
import joblib
joblib.dump(rf, "random_forest_model.pkl")

# 12. Model Evaluation Summary
Summarizes the scores from all trained models for quick comparison.

In [17]:
print("===== Model Accuracies =====")
print(f"Logistic Regression: {logreg_acc:.4f}")
print(f"Best KNN: {knn_acc:.4f}")
print(f"Random Forest: {rf_acc:.4f}")

# 12. Model Evaluation Summary
Summarizes the scores from all trained models for quick comparison.

In [17]:
print("===== Model Accuracies =====")
print(f"Logistic Regression: {logreg_acc:.4f}")
print(f"Best KNN: {knn_acc:.4f}")
print(f"Random Forest: {rf_acc:.4f}")

# 13. Next Steps & Improvements
Suggestions for enhancing the model and workflow in future iterations.

### Possible Improvements:
- Perform cross-validation on all models to get more stable accuracy estimates.
- Normalize or scale continuous features before KNN for better distance computations.
- Try Gradient Boosting or XGBoost for potentially higher accuracy.
- Save preprocessing pipeline to apply the same transforms on future data.
- Perform feature importance analysis for the Random Forest model.